# Phase 10B — Duplicate-Text and Topic-Quality Audit

**Project:** Reddit Narrative Decoder  
**Purpose:** Quantify repeated text within communities and inspect how repeated text could affect topic interpretation.

## Important safeguards
- This notebook does **not** modify the source corpus, model assignments, or existing results.
- It does **not** automatically delete repeated records. Identical text under different IDs may reflect separate source records.
- Text duplication is audited descriptively; it is not proof of a pipeline error.
- The notebook prefers `data/processed/reddit_nlp_corpus.csv` because the Phase 8 merged analysis file may not contain the original text.
- Run this notebook from the project root: `reddit_narrative_decoder`.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()

# If the notebook is opened outside the project root, set PROJECT_ROOT manually:
# PROJECT_ROOT = Path(r"C:\\Users\\TANUJ\\Documents\\reddit_narrative_decoder")

NLP_PATH = PROJECT_ROOT / "data" / "processed" / "reddit_nlp_corpus.csv"
PHASE8_PATH = PROJECT_ROOT / "results" / "tables" / "phase8_cross_community" / "phase8_merged_observations.csv"
OUTPUT_DIR = PROJECT_ROOT / "results" / "tables" / "phase10b_topic_quality_audit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("NLP corpus exists:", NLP_PATH.exists(), NLP_PATH)
print("Phase 8 merged file exists:", PHASE8_PATH.exists(), PHASE8_PATH)
print("Output directory:", OUTPUT_DIR)


In [ ]:
# Load the NLP corpus, which is expected to contain text and community.
if not NLP_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {NLP_PATH}. Run the notebook from the project root "
        "or update PROJECT_ROOT in the setup cell."
    )

corpus = pd.read_csv(NLP_PATH, low_memory=False)
print("Rows:", len(corpus))
print("Columns:", corpus.columns.tolist())
display(corpus.head(3))


In [ ]:
# Select the most appropriate available text column.
text_candidates = ["text", "original_text", "analysis_text", "topic7k_clean_text"]
TEXT_COL = next((col for col in text_candidates if col in corpus.columns), None)

if TEXT_COL is None:
    raise ValueError(
        "No recognizable text column was found. Available columns: "
        + ", ".join(corpus.columns)
    )

if "community" not in corpus.columns:
    if "subreddit" in corpus.columns:
        corpus["community"] = corpus["subreddit"]
    else:
        raise ValueError("No 'community' or 'subreddit' column was found.")

if "record_id" not in corpus.columns:
    if "id" in corpus.columns:
        corpus["record_id"] = corpus["id"].astype(str)
    else:
        raise ValueError("No 'record_id' or 'id' column was found.")

# Normalize whitespace/case for duplicate detection only.
# The original text column is preserved unchanged.
corpus["normalized_text_for_audit"] = (
    corpus[TEXT_COL]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"\\s+", " ", regex=True)
    .str.strip()
)

audit_corpus = corpus[corpus["normalized_text_for_audit"].ne("")].copy()

print("Text column used:", TEXT_COL)
print("Non-empty text rows:", len(audit_corpus))
print("Empty text rows excluded from duplicate-text calculations:",
      len(corpus) - len(audit_corpus))


## 1. Duplicate-text summary by community

`records_beyond_unique_texts` is the number of records above the count of unique normalized text strings in that community. This is a descriptive measure, not a recommendation to remove records.


In [ ]:
duplicate_summary = (
    audit_corpus.groupby("community", dropna=False)
    .agg(
        records=("record_id", "size"),
        unique_normalized_texts=("normalized_text_for_audit", "nunique"),
        unique_record_ids=("record_id", "nunique"),
    )
    .reset_index()
)

duplicate_summary["records_beyond_unique_texts"] = (
    duplicate_summary["records"] - duplicate_summary["unique_normalized_texts"]
)
duplicate_summary["duplicate_text_record_share"] = (
    duplicate_summary["records_beyond_unique_texts"] / duplicate_summary["records"]
)

duplicate_summary = duplicate_summary.sort_values(
    "records_beyond_unique_texts", ascending=False
)

display(duplicate_summary)
duplicate_summary.to_csv(OUTPUT_DIR / "duplicate_text_summary.csv", index=False)
print("Saved:", OUTPUT_DIR / "duplicate_text_summary.csv")


## 2. Inspect repeated-text groups

The table lists normalized text strings that occur more than once within the same community. The record IDs are capped at 10 per group to keep the output manageable.


In [ ]:
duplicate_groups = (
    audit_corpus.groupby(["community", "normalized_text_for_audit"], dropna=False)
    .agg(
        record_count=("record_id", "size"),
        record_ids=("record_id", lambda s: " | ".join(s.astype(str).head(10))),
        example_original_text=(TEXT_COL, "first"),
    )
    .reset_index()
)

duplicate_groups = duplicate_groups[duplicate_groups["record_count"] > 1].copy()
duplicate_groups = duplicate_groups.sort_values(
    ["record_count", "community"], ascending=[False, True]
)

display(duplicate_groups.head(30))
duplicate_groups.to_csv(OUTPUT_DIR / "duplicate_text_groups.csv", index=False)
print("Repeated-text groups:", len(duplicate_groups))
print("Saved:", OUTPUT_DIR / "duplicate_text_groups.csv")


## 3. Exact-text duplication across communities

This checks whether identical normalized text appears in multiple communities. Such overlap can be informative, but it does not by itself establish coordinated activity or common authorship.


In [ ]:
cross_community_groups = (
    audit_corpus.groupby("normalized_text_for_audit")
    .agg(
        record_count=("record_id", "size"),
        community_count=("community", "nunique"),
        communities=("community", lambda s: " | ".join(sorted(set(s.astype(str))))),
        record_ids=("record_id", lambda s: " | ".join(s.astype(str).head(10))),
        example_original_text=(TEXT_COL, "first"),
    )
    .reset_index()
)

cross_community_groups = cross_community_groups[
    (cross_community_groups["record_count"] > 1)
    & (cross_community_groups["community_count"] > 1)
].sort_values(["community_count", "record_count"], ascending=False)

display(cross_community_groups.head(30))
cross_community_groups.to_csv(
    OUTPUT_DIR / "cross_community_duplicate_text_groups.csv", index=False
)
print("Cross-community repeated-text groups:", len(cross_community_groups))
print("Saved:", OUTPUT_DIR / "cross_community_duplicate_text_groups.csv")


## 4. Optional: reconcile duplicate text with Phase 8 and topic assignments

The Phase 8 merged file may contain sentiment/emotion/features but not text. This cell checks its columns and, when possible, joins community/record IDs to the NLP corpus. It does not alter either input.

Topic assignment outputs are model-specific, so do not combine assignments from different model versions without checking their configuration.


In [ ]:
if PHASE8_PATH.exists():
    phase8 = pd.read_csv(PHASE8_PATH, low_memory=False)
    print("Phase 8 rows:", len(phase8))
    print("Phase 8 columns:", phase8.columns.tolist())

    if "record_id" in phase8.columns and "record_id" in corpus.columns:
        phase8_ids = phase8[["record_id"]].drop_duplicates()
        text_ids = corpus[["record_id", "community", TEXT_COL]].drop_duplicates("record_id")
        joined = phase8_ids.merge(text_ids, on="record_id", how="left", indicator=True)
        print("\nPhase 8 record-ID to NLP-text reconciliation:")
        print(joined["_merge"].value_counts(dropna=False).to_string())
        joined.to_csv(OUTPUT_DIR / "phase8_text_id_reconciliation.csv", index=False)
        print("Saved:", OUTPUT_DIR / "phase8_text_id_reconciliation.csv")
    else:
        print("Cannot join: record_id is missing from one of the files.")
else:
    print("Phase 8 merged file not found; skipping optional reconciliation.")


## 5. Interpretation checklist

Use these rules when reviewing the generated CSV files:

1. **Repeated text within a community:** verify whether records are genuinely separate posts/comments or duplicated ingestion. Do not deduplicate solely by matching text.
2. **Repeated text across communities:** document the amount and examples. Treat it as descriptive overlap, not proof of coordination.
3. **Topic prevalence:** the denominator must be stated clearly. Distinguish all corpus records, model-eligible records, assigned records, and `UNASSIGNED` records.
4. **Topic labels:** use representative documents and top terms together. A coherent list of top terms is not enough if representative posts are heterogeneous.
5. **Reporting:** keep weak clusters labelled as generic language, mixed content, or exploratory clusters rather than inventing a specific narrative.
6. **Privacy and ethics:** analyze community-level patterns; do not infer individual mental-health states or sensitive traits.

### Output files
- `duplicate_text_summary.csv`
- `duplicate_text_groups.csv`
- `cross_community_duplicate_text_groups.csv`
- `phase8_text_id_reconciliation.csv` (if the join is possible)

This notebook is read-only with respect to the existing corpus and model results; it only creates audit outputs under `results/tables/phase10b_topic_quality_audit/`.
